Install Libraries

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 103.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install transformers datasets seqeval scikit-learn jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=14bc2c8eaad81b88e13600fba49f4c65903d51306aa0a1f804b324981ed74713
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Imports

In [3]:
import json
import jsonlines
import numpy as np

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

Label Definitions

In [4]:
labels = [
    "O",
    "B-TOPIC","I-TOPIC",
    "B-METHOD","I-METHOD",
    "B-CONCEPT","I-CONCEPT",
    "B-ALGORITHM","I-ALGORITHM",
    "B-OTHER","I-OTHER"
]

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

Load Dataset

In [5]:
data = []

with jsonlines.open("ML-cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

Convert Entities → BIO Labels

In [6]:
def convert_to_tokens(sample):

    text = sample["input"]
    entities = json.loads(sample["output"])

    tokens = text.split()
    tags = ["O"] * len(tokens)

    for ent in entities:

        ent_text = ent["entity"]
        label = ent["label"]

        ent_tokens = ent_text.split()

        for i in range(len(tokens)):

            if tokens[i:i+len(ent_tokens)] == ent_tokens:

                tags[i] = "B-" + label

                for j in range(1,len(ent_tokens)):
                    tags[i+j] = "I-" + label

    return {"tokens":tokens,"ner_tags":tags}

Prepare Dataset

In [7]:
processed = []

for sample in data:
    processed.append(convert_to_tokens(sample))

dataset = Dataset.from_list(processed)

Train/Test Split (90/10)

In [8]:
train_test = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = train_test["train"]
test_dataset = train_test["test"]

Load SciBERT

In [9]:
model_name = "allenai/scibert_scivocab_cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored wh

Tokenization

In [10]:
MAX_LEN = 512

def tokenize(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=MAX_LEN,        # ✅ truncate sequences to 512
        is_split_into_words=True,
    )

    word_ids = tokenized.word_ids()
    labels_ids = []

    for word_id in word_ids:
        if word_id is None:
            labels_ids.append(-100)
        else:
            label = example["ner_tags"][word_id].upper()
            labels_ids.append(label2id.get(label, label2id["O"]))

    tokenized["labels"] = labels_ids
    return tokenized

In [11]:
train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/1334 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Training Arguments

In [12]:
training_args = TrainingArguments(

    output_dir="scibert_ner",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Metrics Function

In [13]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):

        temp_pred = []
        temp_lab = []

        for p_i,l_i in zip(pred,lab):

            if l_i != -100:
                temp_pred.append(id2label[p_i])
                temp_lab.append(id2label[l_i])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {

        "precision": precision_score(true_labels,true_predictions),
        "recall": recall_score(true_labels,true_predictions),
        "f1": f1_score(true_labels,true_predictions)
    }

Trainer

In [14]:
from transformers import DataCollatorForTokenClassification

# This will pad both input_ids and labels to the longest sequence in the batch
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

# Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,  # ✅ ensures consistent batch lengths
    compute_metrics=compute_metrics
)

Train Model

In [15]:
trainer.train()

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.063121,0.790146,0.892784,0.838335
2,No log,0.036151,0.883041,0.934021,0.907816
3,0.086502,0.035135,0.894118,0.940206,0.916583


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=501, training_loss=0.08640779307234787, metrics={'train_runtime': 134.6449, 'train_samples_per_second': 29.723, 'train_steps_per_second': 3.721, 'total_flos': 274557450446760.0, 'train_loss': 0.08640779307234787, 'epoch': 3.0})

Evaluate Model

In [16]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import accuracy_score

# Get predictions (this works regardless of evaluate hooks)
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    temp_pred, temp_lab = [], []
    for p_i, l_i in zip(pred_seq, label_seq):
        if l_i != -100:  # ignore padding
            temp_pred.append(id2label[p_i])
            temp_lab.append(id2label[l_i])
    true_predictions.append(temp_pred)
    true_labels.append(temp_lab)

# Compute metrics
micro_precision = precision_score(true_labels, true_predictions)
micro_recall = recall_score(true_labels, true_predictions)
micro_f1 = f1_score(true_labels, true_predictions)

report = classification_report(true_labels, true_predictions, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]

# Exact accuracy
flat_true, flat_pred = [], []
for t_seq, p_seq in zip(true_labels, true_predictions):
    flat_true.extend(t_seq)
    flat_pred.extend(p_seq)
exact_accuracy = accuracy_score(flat_true, flat_pred)

# Partial accuracy
partial_matches = 0
total_tokens = 0
for t_seq, p_seq in zip(true_labels, true_predictions):
    for t, p in zip(t_seq, p_seq):
        total_tokens += 1
        if t == p:
            partial_matches += 1
        elif t != "O" and p != "O" and t.split("-")[-1] == p.split("-")[-1]:
            partial_matches += 0.5
partial_accuracy = partial_matches / total_tokens

# Print metrics
print("\n===== Evaluation Metrics =====\n")
print("Micro Precision :", round(micro_precision,4))
print("Micro Recall    :", round(micro_recall,4))
print("Micro F1        :", round(micro_f1,4))
print("Macro F1        :", round(macro_f1,4))
print("Exact Accuracy  :", round(exact_accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))
print("\nDetailed Classification Report:\n")
print(classification_report(true_labels, true_predictions))


===== Evaluation Metrics =====

Micro Precision : 0.8941
Micro Recall    : 0.9402
Micro F1        : 0.9166
Macro F1        : 0.8685
Exact Accuracy  : 0.989
Partial Accuracy: 0.989

Detailed Classification Report:

              precision    recall  f1-score   support

   ALGORITHM       0.73      0.80      0.76        10
     CONCEPT       0.93      0.95      0.94       243
      METHOD       0.89      0.72      0.80        46
       OTHER       0.88      0.99      0.93        83
       TOPIC       0.85      0.99      0.91       103

   micro avg       0.89      0.94      0.92       485
   macro avg       0.86      0.89      0.87       485
weighted avg       0.90      0.94      0.92       485

